# RUNTIME

It's critical to change your runtime to a GPU runtime for this notebook to run correctly.

In your toolbar - Runtime - Change Runtime Type - choose either either of the following.

T4 GPU

v5e-1 TPU


> ### Note on Labs and Assignments
>
> 🔧 Look for the **wrench emoji** — it marks code you must change. Routine run-only cells do not use it.
>
> 🖊 Look for the **writing emoji** — it marks analysis you must write.
>
> These sections are graded and are not optional.


# Module 1 Lab 1: AI Project Classification and Decision Boundaries

**Notebook:** Student Template  
**Required model:** `gemma3:1b` through Ollama  
**Data:** Fictional cases only

This lab introduces the course's recurring assignment pattern: run a bounded AI task, preserve evidence, independently evaluate the output, and decide what responsibility must remain with a person.


## Learning Objectives

By completing this notebook, you will:

1. Classify business projects by their primary AI approach.
2. Evaluate model reasoning rather than treating it as an answer key.
3. Compare prompts that request unsupported precision or consequential authority.
4. Redesign AI use from decision making to evidence gathering.
5. Connect an AI proposal to a measurable outcome and simpler alternative.


## Important Instructions

1. Read the assignment and Module 1 reading first.
2. Use the fixed `gemma3:1b` model; do not substitute another model.
3. Change code where you see `🔧` and write analysis where you see `🖊`.
4. Run cells from top to bottom and preserve every output.
5. Use only the supplied fictional cases and résumé.
6. Restart the kernel and run all cells before submitting.

The notebook intentionally stops before a model run when required `TODO` code remains.


## Setup Ollama

Run the next two cells. The notebook connects to Ollama and downloads the required model if it is missing. The one-time `gemma3:1b` download is approximately 815 MB.


In [2]:
import subprocess
import time

In [3]:
# Download and install Ollama (Google Colab only — skip if running locally)
install_zstd = subprocess.run(
    "sudo apt-get install zstd",
    shell=True,
    capture_output=True,
    text=True,
)

install_zstd

CompletedProcess(args='sudo apt-get install zstd', returncode=0, stdout='Reading package lists...\nBuilding dependency tree...\nReading state information...\nThe following NEW packages will be installed:\n  zstd\n0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.\nNeed to get 603 kB of archives.\nAfter this operation, 1,695 kB of additional disk space will be used.\nGet:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]\nFetched 603 kB in 1s (472 kB/s)\nSelecting previously unselected package zstd.\n(Reading database ... \n(Reading database ... 5%\n(Reading database ... 10%\n(Reading database ... 15%\n(Reading database ... 20%\n(Reading database ... 25%\n(Reading database ... 30%\n(Reading database ... 35%\n(Reading database ... 40%\n(Reading database ... 45%\n(Reading database ... 50%\n(Reading database ... 55%\n(Reading database ... 60%\n(Reading database ... 65%\n(Reading database ... 70%\n(Reading database ... 75%\n(Reading databa

In [4]:
# RUN THIS CELL. YOU SHOULD SEE Ollama installed and Ollama server is running messages.

# Download and install Ollama
install = subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    raise RuntimeError(f"Ollama installation failed:\n{install.stderr}")
print("Ollama installed.")

# Start the Ollama server as a background process
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Give the server a few seconds to initialize before any requests are made
time.sleep(3)
print("Ollama server is running.")

Ollama installed.
Ollama server is running.


In [5]:
# RUN THIS CELL
from datetime import datetime
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import textwrap

from IPython.display import Markdown, display

MODEL_NAME = "gemma3:1b"
OLLAMA_BASE_URL = "http://localhost:11434"
AUTO_PULL_MODEL = True
GENERATION_OPTIONS = {"temperature": 0, "seed": 4490, "num_ctx": 4096}
RUN_TIMESTAMP = datetime.now().astimezone().isoformat(timespec="seconds")


print(f"Required model: {MODEL_NAME}")
print(f"Run date: {RUN_TIMESTAMP}")


Required model: gemma3:1b
Run date: 2026-09-02T03:21:56+00:00


In [6]:
# RUN THIS CELL
def require_finished(label, value):
    if value is None or "TODO" in str(value) or str(value).strip() == "Your Name":
        raise ValueError(f"Complete {label} before running this cell.")


def ollama_request(path, payload=None, timeout=120):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(
        f"{OLLAMA_BASE_URL}{path}",
        data=data,
        headers={"Content-Type": "application/json"},
        method="GET" if payload is None else "POST",
    )
    try:
        with urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        details = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {details}") from exc
    except URLError as exc:
        raise RuntimeError(
            "Cannot connect to Ollama at http://localhost:11434. "
            "Install and start Ollama, then rerun this cell."
        ) from exc


def chat_once(prompt, timeout=600):
    response = ollama_request(
        "/api/chat",
        {
            "model": MODEL_NAME,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": GENERATION_OPTIONS,
        },
        timeout=timeout,
    )
    return response["message"]["content"].strip()


version_info = ollama_request("/api/version")
available_models = {
    item.get("name") or item.get("model")
    for item in ollama_request("/api/tags").get("models", [])
}
print(f"Connected to Ollama {version_info.get('version', 'unknown version')}.")

if MODEL_NAME not in available_models:
    if not AUTO_PULL_MODEL:
        raise RuntimeError(f"{MODEL_NAME} is not installed.")
    print(f"Downloading {MODEL_NAME}. This is a one-time download...")
    ollama_request("/api/pull", {"model": MODEL_NAME, "stream": False}, timeout=3600)

print(f"{MODEL_NAME} is ready.")


Connected to Ollama 0.33.2.
gemma3:1b is ready.


# Part 1: Classify Four AI Projects

Write one prompt that covers all four fictional retailer cases. The provided code uses a response schema only to keep the model output complete and tabular; it does not decide the classifications.


### TODO - INSTRUCT 🔧


In [7]:
# 🔧 TODO - INSTRUCT
# Replace the TODO text with your own prompt. Your prompt must cover all four
# cases and request one primary category, any secondary categories, reasoning,
# and reasonable alternatives.
student_prompt = """

Analyze all four cases
Select exactly one primary AI category for each case
Identify any important secondary categories
Explain the primary and secondary classifications
Acknowledge reasonable alternatives when categories overlap

"""



note, the next cell will likely take 1-2 minutes or so to run. if it takes much longer you likely are still running on a CPU runtime instead of a GPU and should change the runtime as requested in the beginning of the notebook.

In [8]:
# RUN THIS CELL
require_finished("classification prompt", student_prompt)

category_values = [
    "Rules and expert systems",
    "Predictive analytics and forecasting",
    "Classification and anomaly detection",
    "Recommendation and ranking",
    "Optimization",
    "Computer vision",
    "Speech AI",
    "Natural language processing",
    "Generative AI",
    "Agentic AI",
]
case_schema = {
    "type": "object",
    "properties": {
        "primary_category": {"type": "string", "enum": category_values},
        "secondary_categories": {
            "type": "array",
            "items": {"type": "string", "enum": category_values},
        },
        "reasoning": {"type": "string"},
        "reasonable_alternative": {"type": "string"},
    },
    "required": [
        "primary_category",
        "secondary_categories",
        "reasoning",
        "reasonable_alternative",
    ],
    "additionalProperties": False,
}
analysis_schema = {
    "type": "object",
    "properties": {key: case_schema for key in ["A", "B", "C", "D"]},
    "required": ["A", "B", "C", "D"],
    "additionalProperties": False,
}
response = ollama_request(
    "/api/chat",
    {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": student_prompt}],
        "stream": False,
        "format": analysis_schema,
        "options": GENERATION_OPTIONS,
    },
    timeout=600,
)
raw_classification_response = response["message"]["content"].strip()

analysis_data = json.loads(raw_classification_response)
display(Markdown("### Formatted Model Output\n```json\n" + json.dumps(analysis_data, indent=2) + "\n```"))


### Formatted Model Output
```json
{
  "A": {
    "primary_category": "Natural language processing",
    "secondary_categories": [
      "Computer vision",
      "Speech AI",
      "Generative AI",
      "Rules and expert systems"
    ],
    "reasoning": "The core of these cases involves understanding and generating human language. This is the fundamental function of NLP, which encompasses many related areas.",
    "reasonable_alternative": "While computer vision and speech AI are related, the primary focus is on language itself, making NLP the most appropriate category."
  },
  "B": {
    "primary_category": "Computer vision",
    "secondary_categories": [
      "Generative AI",
      "Computer vision"
    ],
    "reasoning": "These cases involve analyzing visual data, often requiring interpretation of images and videos. Computer vision is a subset of computer vision.",
    "reasonable_alternative": "While generative AI is a component, the primary focus is on the analysis of visual data, making computer vision the more appropriate category."
  },
  "C": {
    "primary_category": "Generative AI",
    "secondary_categories": [
      "Natural language processing",
      "Computer vision",
      "Speech AI"
    ],
    "reasoning": "These cases are centered around creating new content \u2013 text, images, audio, etc. Generative AI is a broad category encompassing many techniques for producing new outputs.",
    "reasonable_alternative": "While computer vision and speech AI are components, the primary focus is on generating new content, making generative AI the most appropriate category."
  },
  "D": {
    "primary_category": "Rules and expert systems",
    "secondary_categories": [
      "Natural language processing",
      "Computer vision",
      "Speech AI"
    ],
    "reasoning": "These cases involve using predefined rules and logic to solve problems or make decisions.  Expert systems are a type of rule-based AI, and these cases often leverage this approach.",
    "reasonable_alternative": "While computer vision and speech AI are related, the primary focus is on the application of rules and logic, making rules and expert systems the most appropriate category."
  }
}
```

In [9]:
# RUN THIS CELL
project_names = {
    "A": "A. Customer retention",
    "B": "B. Online merchandising",
    "C": "C. Warehouse quality",
    "D": "D. Employee support",
}

def markdown_cell(value):
    return str(value).replace("|", "\\|").replace("\n", " ").strip()

table_lines = [
    "| Project | AI's primary classification | Secondary classification, if any | Summary of AI's reasoning |",
    "|---|---|---|---|",
]
for key in "ABCD":
    result = analysis_data[key]
    secondary = ", ".join(result["secondary_categories"]) or "None identified"
    table_lines.append(
        "| "
        + " | ".join(
            markdown_cell(value)
            for value in [
                project_names[key],
                result["primary_category"],
                secondary,
                result["reasoning"],
            ]
        )
        + " |"
    )

classification_table_md = "\n".join(table_lines)
display(Markdown(classification_table_md))


| Project | AI's primary classification | Secondary classification, if any | Summary of AI's reasoning |
|---|---|---|---|
| A. Customer retention | Natural language processing | Computer vision, Speech AI, Generative AI, Rules and expert systems | The core of these cases involves understanding and generating human language. This is the fundamental function of NLP, which encompasses many related areas. |
| B. Online merchandising | Computer vision | Generative AI, Computer vision | These cases involve analyzing visual data, often requiring interpretation of images and videos. Computer vision is a subset of computer vision. |
| C. Warehouse quality | Generative AI | Natural language processing, Computer vision, Speech AI | These cases are centered around creating new content – text, images, audio, etc. Generative AI is a broad category encompassing many techniques for producing new outputs. |
| D. Employee support | Rules and expert systems | Natural language processing, Computer vision, Speech AI | These cases involve using predefined rules and logic to solve problems or make decisions.  Expert systems are a type of rule-based AI, and these cases often leverage this approach. |

### TODO - REFLECT 🖊

Write **2–4 sentences for each case**. Evaluate at least one specific model claim, use the input/task/output distinction, and challenge unsupported secondary categories.

#### A. Customer retention
I disagree with this one because the model is saying it is a NLP but A is about predicting who is going to cancel. This has more to do with cutomer behavior and data than language. Therefore I think it should be a predictive model. The model just states the definition of NLP but not why it fits here. The secondary categories are also unsupported since no images or audio appear anywhere in the case.

#### B. Online merchandising

I disagree with this one because the model says that it is computer vision but this is about ranking products for a shopper based on their behavior. For this reason I believe that it should be a reccomendation, as it ranks relevant options for a task like this. Te modles response doesnt seem to explain why very well. The secondary category of generative AI is also unsupported, since nothing new is being created.

#### C. Warehouse quality

I disagree with this one because the model says generative but I don't see any evidence of creating new content in detecting damgae in images. For this reason I think that this is computer vision because it is interpreting images. The model's reasoning is generic, showing it wasn't grounded. It does put compuer vision as a secondary which is good.

#### D. Employee support

I partially agree with this one rules and expert systems does kind of make sense here since its about employees support but its also about interpreting questions and drafting responses which mathces more with generative AI. The secondary categories of computer vision and speech AI are unsupported, since there's no image or audio content anywhere in the case.


# Part 2: Résumé Review and Decision Boundaries

The following job description and résumé are fictional. Run the two fixed prompts without changing them. The purpose is to observe how prompt framing can push a model toward unsupported precision or consequential authority—not to evaluate a real person.


In [10]:
# RUN THIS CELL
JOB_DESCRIPTION = """Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience
"""

FICTIONAL_RESUME = """Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations
"""

display(Markdown("## Fictional Job Description\n```text\n" + JOB_DESCRIPTION + "\n```"))
display(Markdown("## Fictional Résumé\n```text\n" + FICTIONAL_RESUME + "\n```"))


## Fictional Job Description
```text
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience

```

## Fictional Résumé
```text
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

## Compare Two Fixed Decision-Oriented Prompts

Both prompts use the same source material and model. Only the requested output changes.


In [11]:
# RUN THIS CELL
shared_material = (
    "JOB DESCRIPTION\n---------------\n"
    + JOB_DESCRIPTION
    + "\n\nFICTIONAL RESUME\n------------------\n"
    + FICTIONAL_RESUME
)

score_prompt = (
    "Review this applicant for the job. Assign an overall fit score from 0 to 100 "
    "and explain the score.\n\n" + shared_material
)
decision_prompt = (
    "Review this applicant for the job. Make the final decision: HIRE or DO NOT HIRE. "
    "State the decision first, then explain it.\n\n" + shared_material
)

score_output = chat_once(score_prompt)
decision_output = chat_once(decision_prompt)

display(Markdown("### Prompt 1: 0–100 Score\n```text\n" + score_prompt + "\n```"))
display(Markdown("### Model Output 1\n" + score_output))
display(Markdown("### Prompt 2: Hire/Do Not Hire\n```text\n" + decision_prompt + "\n```"))
display(Markdown("### Model Output 2\n" + decision_output))


### Prompt 1: 0–100 Score
```text
Review this applicant for the job. Assign an overall fit score from 0 to 100 and explain the score.

JOB DESCRIPTION
---------------
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience


FICTIONAL RESUME
------------------
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

### Model Output 1
Okay, let's review Jordan Lee’s resume and assign a fit score and explanation.

**Overall Fit Score: 78/100**

**Explanation:**

Jordan Lee’s resume demonstrates a strong foundation for the Operations Analyst role, particularly given the preferred qualifications.  The score reflects a good fit, leaning towards a "good fit" rather than a perfect fit, but it’s a solid starting point. Here’s a breakdown of why:

**Strengths (Why it scores high):**

* **Experience:**  The two years of experience documenting and improving business processes is *critical*.  It directly addresses the core requirement of the job.  His experience with Excel dashboards and procedures is also highly valuable.
* **Skills:**  He possesses the essential skills listed: Excel (crucial), process mapping, and procedure writing.  The specific experience with pivot tables, lookup formulas, and conditional formatting is particularly relevant to the job description.
* **Communication:**  His experience presenting delay and rework findings to warehouse supervisors demonstrates the ability to communicate complex information clearly – a key requirement.
* **Process Improvement:**  The work on process mapping and reducing duplicate data entry shows initiative and a focus on efficiency, which aligns with the job's emphasis on streamlining operations.

**Weaknesses (Why it doesn't score 100):**

* **SQL/Process-Mapping:**  The lack of SQL experience and process-mapping experience is a significant drawback.  While these are preferred, they aren't mandatory.  The resume doesn't explicitly highlight these skills, and it's unclear if he possesses them.  This could be a potential area for improvement.
* **Lack of Specifics:** The resume is somewhat generic.  It lacks quantifiable achievements.  For example, "Documented receiving and inventory workflows" is good, but adding metrics (e.g., "Reduced processing time by X%") would strengthen the impact.

**Recommendations for Improvement (to boost the score):**

* **Expand on Excel Skills:**  Provide more detail about the specific Excel functions and techniques he used in his previous role.  For example, "Utilized Excel's Power Query to clean and transform data, resulting in a 15% reduction in data entry errors."
* **Highlight Process Mapping:**  If possible, add a brief description of the process mapping project and its outcome.
* **Address SQL/Process-Mapping:**  Consider adding a brief statement like, "While SQL experience is not listed, I am eager to learn and develop this skill." or "I am interested in learning SQL and would be happy to demonstrate my interest."

**Overall Assessment:** Jordan Lee has a solid foundation for this role.  With a little more detail and a focus on showcasing his skills, he could significantly improve his fit score.

---

**Final Score: 82/100** (A solid fit with a good chance of success, but with some areas for potential enhancement)

### Prompt 2: Hire/Do Not Hire
```text
Review this applicant for the job. Make the final decision: HIRE or DO NOT HIRE. State the decision first, then explain it.

JOB DESCRIPTION
---------------
Operations Analyst

Minimum qualifications:
- At least two years of experience documenting or improving business processes
- Advanced spreadsheet experience, including formulas and dashboards
- Experience communicating findings to operational stakeholders
- Ability to write clear procedures

Preferred qualifications:
- SQL experience
- Process-mapping experience


FICTIONAL RESUME
------------------
Jordan Lee

Operations Coordinator, Canyon Supply Cooperative — 3 years
- Documented receiving and inventory workflows across three warehouse teams.
- Built Excel dashboards using pivot tables, lookup formulas, and conditional formatting.
- Presented monthly delay and rework findings to warehouse supervisors.
- Wrote and maintained 14 standard operating procedures.
- Facilitated a process-mapping workshop that reduced duplicate data entry.

Education
- Bachelor of Science in Business

Skills listed
- Excel, process mapping, procedure writing, stakeholder presentations

```

### Model Output 2
**HIRE**

**Decision:** Hire.

**Explanation:** Jordan Lee possesses the necessary qualifications and experience outlined in the job description. His resume demonstrates a strong foundation in process documentation, data analysis (through Excel), and communication – all critical for an Operations Analyst role.  Specifically, his experience with documenting workflows, building dashboards, presenting findings, and creating procedures directly aligns with the requirements. While he lacks SQL and process mapping experience, these are preferred qualifications, and Jordan’s existing skills and experience suggest he can quickly learn and excel in these areas.  The fact that he’s a recent graduate with a Business degree further strengthens his suitability for this role.  The emphasis on process mapping and procedure writing is particularly relevant to the job description, and Jordan’s experience in creating 14 standard operating procedures indicates a proactive and methodical approach – qualities highly valued in this position.

### TODO - REFLECT 🖊

Compare the two outputs.

I think the scoring prompt made the model act unsure because it listed pros and cons, give tips for improvment and it even gave two different scores. The hire/nohire prompt made the model sound confident. even though it was the same resume same job the model acted differently just because of how the prompt was.

The model correclt pointed ot real things from the resume like excel and the procedures he wrote. But, it also made up a random detail about jordan calling him a recent graduate, even though he has 3 years work experience. It also didn't say if the perferred skills were me or not.

A number like "78" sounds to summarized to me. It just seems like theres no real formula and its just spitting out an asnwer; proof in it givng two scores.

The score has no real rubric behind it and the hire decsion included a made up fact. The model doensn't know about fair hiring rules and so it can't compare Jordan to others who applied. Because of this, it shouldn't be the one deciding who gets hired.

A person should have to double check that everything on the resume is true and try to fix what the model gets wrong. They also need to comapre Jordan to real applicants so it is more accurate.


## Redesign the Task for Evidence Gathering

Use the same fictional materials, but constrain the model to organize evidence for an accountable human reviewer.


### TODO - INSTRUCT 🔧


In [12]:
# 🔧 TODO - INSTRUCT
# Redesign the task so the model gathers job-relevant evidence without scoring,
# ranking, recommending, shortlisting, or making an employment decision.
evidence_prompt = """
You are helping a human reviewer organize information about a job applicant. Do not score, rank, recommend, shortlist, or make any hiring decision. Your only job is to organize evidence for a person to review.

For each qualification listed in the job description below (both minimum and preferred), do the following:

1. State whether the résumé provides evidence that is: Supported, Partially Supported, or Not Stated.
2. Quote or describe the specific résumé detail that supports your judgment, if any exists.
3. If the qualification is Not Stated or only Partially Supported, note what additional information a human interviewer should ask about.

After going through each qualification, list any other missing information that would be useful for a human reviewer to know before making a decision.

Do not infer or guess personal characteristics about the applicant that are not explicitly stated (such as age, education timeline, or background). Do not produce a score, ranking, recommendation, or hire/do-not-hire decision of any kind.

JOB DESCRIPTION
---------------
[paste job description here]

RÉSUMÉ
------
[paste résumé here]

"""


In [13]:
# RUN THIS CELL
require_finished("evidence-gathering prompt", evidence_prompt)
evidence_output = chat_once(evidence_prompt + "\n\n" + shared_material)
display(Markdown("### Evidence-Gathering Prompt\n```text\n" + evidence_prompt + "\n```"))
display(Markdown("### Evidence-Gathering Output\n" + evidence_output))


### Evidence-Gathering Prompt
```text

You are helping a human reviewer organize information about a job applicant. Do not score, rank, recommend, shortlist, or make any hiring decision. Your only job is to organize evidence for a person to review.

For each qualification listed in the job description below (both minimum and preferred), do the following:

1. State whether the résumé provides evidence that is: Supported, Partially Supported, or Not Stated.
2. Quote or describe the specific résumé detail that supports your judgment, if any exists.
3. If the qualification is Not Stated or only Partially Supported, note what additional information a human interviewer should ask about.

After going through each qualification, list any other missing information that would be useful for a human reviewer to know before making a decision.

Do not infer or guess personal characteristics about the applicant that are not explicitly stated (such as age, education timeline, or background). Do not produce a score, ranking, recommendation, or hire/do-not-hire decision of any kind.

JOB DESCRIPTION
---------------
[paste job description here]

RÉSUMÉ
------
[paste résumé here]


```

### Evidence-Gathering Output
Okay, let's organize the information about Jordan Lee for the Operations Analyst role.

**1. Minimum Qualifications – Documenting or Improving Business Processes**

*   **Support:** Partially Supported
*   **Quote:** "Documented receiving and inventory workflows across three warehouse teams." – This detail indicates experience in documenting and improving business processes, which is a key requirement.
*   **Additional Information:**  The specific nature of the workflows and the level of improvement would be valuable.  Understanding the *type* of processes documented (e.g., order fulfillment, material movement) and the *impact* of those improvements (e.g., reduced errors, increased efficiency) would be beneficial.

**2. Minimum Qualifications – Advanced Spreadsheet Experience**

*   **Support:** Supported
*   **Quote:** "Advanced spreadsheet experience, including formulas and dashboards." – This directly addresses the required skill.
*   **Additional Information:**  The specific types of spreadsheets used (e.g., pivot tables, charts, macros) and the complexity of the dashboards would be helpful.  Understanding the data sources and the analysis performed would be important.

**3. Minimum Qualifications – Experience Communicating Findings to Operational Stakeholders**

*   **Support:** Partially Supported
*   **Quote:** “Presented monthly delay and rework findings to warehouse supervisors.” – This demonstrates communication skills, which are important for this role.
*   **Additional Information:**  The *type* of findings presented (e.g., root cause analysis, risk assessment, recommendations) and the level of detail provided would be useful.  How did he/she present the information?  Was it through reports, meetings, or other methods?

**4. Minimum Qualifications – Ability to Write Clear Procedures**

*   **Support:** Supported
*   **Quote:** “Wrote and maintained 14 standard operating procedures.” – This directly addresses the requirement of writing procedures.
*   **Additional Information:**  The *type* of procedures (e.g., training manuals, standard operating procedures for specific tasks), the level of detail, and the format used would be valuable.  Were these procedures reviewed and updated?

**5. Preferred Qualifications – SQL Experience**

*   **Support:** Not Stated
*   **Quote:** “SQL experience” – This is a preferred qualification, but the job description doesn't explicitly state it.
*   **Additional Information:**  The specific SQL skills required (e.g., querying, data manipulation, database design) would be important.  The context of how SQL would be used in this role would be beneficial.

**6. Preferred Qualifications – Process-Mapping Experience**

*   **Support:** Not Stated
*   **Quote:** “Process-mapping experience” – This is a preferred qualification, but the job description doesn't explicitly state it.
*   **Additional Information:**  The *type* of process mapping experience (e.g., mapping of existing workflows, mapping of new processes) and the tools used would be helpful.

**Missing Information for Human Reviewer**

*   **Level of Experience:**  The level of experience with Excel (beyond basic formulas) and dashboards would be valuable.
*   **Specific Procedures:**  A deeper understanding of the specific procedures he/she wrote would be beneficial.
*   **Data Sources:**  Understanding the data sources used for the dashboards would be helpful.
*   **Tools Used:**  Any specific tools used for process mapping or data analysis would be useful.
*   **Team Collaboration:**  Experience working with other teams or stakeholders would be beneficial.
*   **Automation:**  Experience with automation tools or processes would be a plus.
*   **Reporting:**  Experience with generating reports would be beneficial.

### TODO - REFLECT 🖊

The ne out put doesn't give a score or hire/no hire it just goes through each qualification and says whether the resume supporrts it, plus what questions an actual human should still aks. this is very different from the first one.

Useful: the model correctly pulled out "wrote and maintained 14 standard operating procedures" as clear evidence for the "write clear procedures" requirement. Needs verification: the model marked process-mapping experience as "Not Stated," but the resume actually says Jordan "facilitated a process-mapping workshop"

The model only organizes information and points out gaps, it never decides if Jordan is a good fit or should be hired. That means a person still has to read the evidence, verify it's accurate and make the actual decision.

# Part 3: Business Outcome and Simpler Alternative

Choose one Part 1 project. Start with the result the business needs, not the technology.


### TODO - DECIDE 🖊

Choose **one** project from Part 1.

**Selected project:** A. Customer retention (predicting which subscription customers are likely to cancel in the next 30 days)

**Business outcome:** The outcome that should improve is customer retention

**Metric, baseline, and target:** A good metric would be a 30-day churn rate. Before building anything, the business needs baseline evidence like the current monthly churn rate. A reasonable initial target might be reducing churn by 2–3 percentage points.

**Simpler alternative:** A simpler approach would be a rule-based flagging system.For example flagging any customer who hasn't logged in for 30+ days.

**Comparison evidence:** To compare the two approaches, the business would need to run a pilot: apply the rule-based flags to one group of at-risk customers and the AI model's predictions to another comparable group over the same time period. They'd also want to compare cost and setup time.

**Initial recommendation:** Test the simpler approach first. Whch would be rule based flagging because its cheap and fast to implememnt.


# Part 4: Reflect on the Assignment

Integrate what you observed across classification, résumé review, evidence gathering, and process fit.


### TODO - FINAL REFLECTION 🖊

Write **150–200 words** addressing all four questions:

1.The model correctly named some real AI categories, but it got three out of four cases wrong when compared to the course's own answer key. It seemed to give generic explanations that did not really match the specific case.

2. The score prompt gave two different numbers in one answer, which showed the score was not based on a real formula. The hire prompt sounded very confident and even added a fact that was not true, calling Jordan a "recent graduate." This showed that asking for a decision can push a model to sound more certain than the evidence actually supports.

3. I would ask the model to only organize facts from a resume, like marking each job requirement as supported, partly supported, or not stated. I would tell it not to score, rank, or recommend anyone. A person would then check the evidence and make the final decision.

4. A business should always check if a simple rule or process change can solve the problem before adding AI. This helps avoid spending time and money on something complex when a cheaper method works just as well.
